This notebook is responsible for generating COCO RLE ANNOTATIONS from spheric, color-masks, it is run SEPARATELY to creating images for dataset, and lasts veeery long.
There are many assumptions about processing masks, in respect to min_area_size, smoothing mask, retrieving pixel value.

In [2]:
import os
from pathlib import Path

os.getcwd()

'c:\\Users\\WA\\Desktop\\badanie\\syncity3D\\jupyter'

In [3]:
import os
import re
import json
import shutil
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
from PIL import Image
from skimage.measure import label, regionprops
from pycocotools import mask as mask_utils
from tqdm import tqdm
from skimage.segmentation import expand_labels


In [85]:
import cv2

@dataclass(frozen=True)
class ClassDef:
    name: str
    category_id: int



WORKSPACE_DIR = Path(r"C:\Users\WA\Desktop\badanie") # !! directory with our project and other projects (ESRI.lib)
BASE_DIR = WORKSPACE_DIR / "syncity3D" 
SCRIPTS_DIR = BASE_DIR / "scripts"

# stage 1 - snapshots back, right, front, left, top, bottom 
CE_SNAPSHOTS_DIR = BASE_DIR / "images"
# stage 2 - spheric 'photos' and their segmention/semantic color masks
SPHERIC_PHOTOS_DIR = BASE_DIR / "dataset_stages" / "spheric_photos_and_masks"
# final stage - COCO format ready dataset /images /annotations
COCO_DATASET_DIR = BASE_DIR / "COCO_DATASETS" / "D4" # D4!!!!!1


CITY_PARAMETERS_DIR = COCO_DATASET_DIR / "city_parameters.json"

with open (CITY_PARAMETERS_DIR, mode="r") as f:
    CITY_PARAMETERS = json.load(f)

#print(CITY_PARAMETERS)
PARAMETERS = CITY_PARAMETERS["parameters"]

# Ile klas?? tyle ile jest parametrów w mieście czy wszystkie możliwości?
#CLASS_DICT = {f"{param_value}": id for id, param_value in enumerate(np.arange(start=0, stop=255, step=1), start=1)}
# PARAM VALUE -> PIXEL VALUE
CLASS_DICT = {f"{param_value}": id for id, param_value in enumerate([int(param*100) for param in PARAMETERS], start=1)}

# OLD
#CLASSES = [ ClassDef("Facade", 1000), ClassDef("Window", 1001) ] + [ClassDef(key, CLASS_DICT[key]) for key in CLASS_DICT.keys()]

CLASSES_M1 =  [ClassDef(key, CLASS_DICT[key]) for key in CLASS_DICT.keys()]
CLASSES_M2 = [ClassDef("0_1000", 1000)] +  [ClassDef(f"1_{key}", CLASS_DICT[key]) for key in CLASS_DICT.keys()] 
# id = 1000 to distinct facade from parameter list, detectron will apply mapping
# 0_* - Facade, 1_* - Window

#print("CLASSES")
#print(CLASSES_M1)
#print(CLASSES_M2)


def get_class_id_per_value(parameter_value: float) -> int:
    return CLASS_DICT.get(f"{parameter_value}", int(parameter_value))


def key_from_filename(fname: str, prefix: str) -> str:
    """
    Extract id from {prefix}\_SynCity3D\_\<id\>\.\<ext\> -> \<id\> ??
    """
    base = os.path.basename(fname)
    if not base.startswith(prefix):
        raise ValueError(f"File {base} does not start with {prefix}")
    stem = os.path.splitext(base)[0]
    return stem.split("_")[-1].split("\.")[0]


def find_pairs(raw_dir: str) -> List[Tuple[str, str, str, str]]:
    """
    Returns list of (key, img_path, mask_path_m1, mask_path_m2)
    """

    imgs = []
    masks_m1 = []
    masks_m2 = []

    for f in os.listdir(raw_dir):
        if f.startswith("m-Normal"):
            imgs.append(f)
        elif f.startswith("m-1"):
            masks_m1.append(f)
        elif f.startswith("m-2"):
            masks_m2.append(f)

    img_map = {key_from_filename(f, "m-Normal"): os.path.join(raw_dir, f) for f in imgs}
    mask_map_m1 = {key_from_filename(f, "m-1"): os.path.join(raw_dir, f) for f in masks_m1}
    mask_map_m2 = {key_from_filename(f, "m-2"): os.path.join(raw_dir, f) for f in masks_m2}

    keys = sorted(set(img_map.keys()) & set(mask_map_m1.keys()) & set(mask_map_m2.keys()))
    
    if not keys:
        raise RuntimeError("No matching x_<id>_<angle> and y_<id>_<angle> pairs found.")

    triad = [(k, img_map[k], mask_map_m1[k], mask_map_m2[k]) for k in keys]
    return triad


def instances_from_binary_mask(bin_mask: np.ndarray, min_area: int = 1) -> List[np.ndarray]:
    """
    Split a binary mask into instance masks using connected components.
    Returns list of HxW boolean masks.
    """
    lab = label(bin_mask.astype(np.uint8), connectivity=2)
    inst = []
    for r in regionprops(lab):
        if r.area < min_area:
            continue
        m = (lab == r.label)
        inst.append(m)
    return inst


def clean_rgb_mask(mask_rgb: np.ndarray) -> np.ndarray:
    '''
    In no case we mix 3 channels at once, so we can cut pixels which match this condition (these are potential artefacts - lot outlines)

    mask_rgb: Color
    '''
    M = mask_rgb.astype(np.uint8)
    R = M[:, :, 0]
    G = M[:, :, 1]
    B = M[:, :, 2]

    artefacts = (R > 0) & (G > 0) & (B > 0)
    binary_mask = np.stack([artefacts, artefacts, artefacts], axis=2)
    mask_rgb[binary_mask] = 0   

    return mask_rgb


def filter_channels(mask_rgb: np.ndarray, channels = "rgb") -> np.ndarray:
    """
    Keep only pixels that are 'almost' pure channel
    Everything else is set to background [0,0,0].

    """

    if mask_rgb.ndim == 2:
        mask_rgb = np.repeat(mask_rgb[:, :, None], 3, axis=2)

    main_min_treshold = 40
    side_max_treshold = 10

    M = mask_rgb.astype(np.uint8)
    R = M[:, :, 0]
    G = M[:, :, 1]
    B = M[:, :, 2]

    # 'pure' -> TOLERANCE, we already 'cleaned' color (segmentation) masks
    pure_red    = (R >= main_min_treshold) & (G < side_max_treshold) & (B < side_max_treshold)
    pure_green  = (G >= main_min_treshold) & (B < side_max_treshold) & (R < side_max_treshold)
    pure_blue   = (B >= main_min_treshold) & (R < side_max_treshold) & (G < side_max_treshold)

    out = np.zeros_like(M)

    if "r" in channels:
        out[pure_red, 0] = R[pure_red]  # keep original R pixels

    # we may keep green channel too, as some facades can be extremely wiiide, so it w/h ratio exceeds 2.55 - FORMER IDEA
    if "g" in channels:
        out[pure_green, 1] = G[pure_green]

    if "b" in channels:
        out[pure_blue, 2] = B[pure_blue]

    #Image.fromarray(out).show()

    return out


def coco_rle_from_mask(mask: np.ndarray) -> Dict:
    """
    Use pycocotools to produce COCO RLE.
    mask must be Fortran-contiguous (column-major) for correct encoding.
    """
    m = np.asfortranarray(mask.astype(np.uint8))
    rle = mask_utils.encode(m)
    # pycocotools returns counts as bytes; convert to utf-8 string for JSON
    rle["counts"] = rle["counts"].decode("utf-8")
    
    return rle


def bbox_from_mask(mask: np.ndarray) -> List[float]:
    ys, xs = np.where(mask)
    x0, x1 = xs.min(), xs.max()
    y0, y1 = ys.min(), ys.max()
    return [float(x0), float(y0), float(x1 - x0 + 1), float(y1 - y0 + 1)]


def get_parameter_value_from_instance_mask(instance_binary_mask: np.ndarray, rgb_mask: np.ndarray) -> float:
    """ 
    Returns parameter value (pixel value).

    Reads every pixel value occuring in rgb_mask of an INSTANCE (of either a facade or a window), 
    which is a 3-channel array (where G and B channels are arrays filled with zeros), counts them
    and selects the most likely one (the one which appeared the most times).
    """

    #instance_mask = np.repeat(instance_binary_mask[:, :, None], 3, axis=2)
    #instance_mask = np.stack([instance_binary_mask, instance_binary_mask, instance_binary_mask], axis=2)

    # mask with dark pixels
    #instance_idx = (instance_binary_mask == 0)
    rgb_mask_filtered = rgb_mask.copy()
    rgb_mask_filtered[~instance_binary_mask] = 0

    #Image.fromarray(instance_binary_mask).show()
    #Image.fromarray(rgb_mask_filtered).show()

    # sum because proportion parameter can comprise R and G channel in M1, in M2 channels are separated, so it doesn't affect anything
    #accumulated_mask = np.sum(rgb_mask_filtered, axis=2)
    M = rgb_mask_filtered.astype(np.uint8)
    R = M[:, :, 0]

    values, counts = np.unique(R, return_counts=True)

    #Image.fromarray(instance_binary_mask).show()
    #Image.fromarray(rgb_mask_filtered).show()

    #print()
    #print("VALUES:", values)
    #print("COUNTS:", counts)

    #skip dark instances
    if len(values) == 1 & values[0] == 0:
        return -1 

    most_common_value = np.argmax(counts[1:]) # we exclude dark/mask (0) at the start
    true_value = values[most_common_value + 1]

    #print("MOST COMMON VALUES:", most_common_value)
    #print("TRUE PARAM VALUE:", true_value)

    match = find_closest_match(true_value)
    #print(f"CLOSEST MATCH TO {true_value} is {match}")

    return int(match)


def find_closest_match(true_value):
    #print(PARAMETERS)
    distances = [abs(true_value-param * 100) for param in PARAMETERS]
    #print("Distances", distances)
    min_distance = np.argmin(np.array(distances))
    match = PARAMETERS[min_distance]
    return match * 100 # return pixel value


def smooth_binary_mask(instance_binary_mask: np.ndarray) -> np.ndarray:
    '''
    Smoothing BINARY masks, because initial binary mask suffer heavily from aliasing in previous step (any value in channel > 0),
    also rendering in CityEngine yields some artifacts.

    We basically apply basic smoothering  and expansion
    '''
    #Image.fromarray(instance_binary_mask).show()

    kernel = np.ones((7,7), np.uint8)

    mask_cv2 = (instance_binary_mask * 255).astype(np.uint8)
    cleaned = cv2.morphologyEx(mask_cv2, cv2.MORPH_CLOSE, kernel)
    

    # expand_labels grows labels outward
    expanded = expand_labels(cleaned, distance=2)
    
    #Image.fromarray(expanded).show()

    return expanded


def display_contours(binary_mask, contour_instances):
    mask_contours = np.zeros(shape=binary_mask.shape)
    for binary_contour_mask in contour_instances:
        mask_contours += binary_contour_mask

    binary_mask_contours = (mask_contours > 0)
    contours = np.zeros(shape=binary_mask_contours.shape)
    contours[binary_mask_contours] = 255
    Image.fromarray(contours).show()


def export_split_to_coco(
    pairs: List[Tuple[str, str, str, str]],
    out_images_dir: str,
    out_json_m1_path: str,
    out_json_m2_path: str,
    copy_images: bool = False,
    min_facade_area: int = 100,
    min_window_area: int = 100
):
    os.makedirs(out_images_dir, exist_ok=True)
    os.makedirs(os.path.dirname(out_json_m1_path), exist_ok=True)
    os.makedirs(os.path.dirname(out_json_m2_path), exist_ok=True)
    
    images = []
    annotations_m1 = []
    annotations_m2 = []

    #categories = [{"id": c.category_id, "name": c.name, "supercategory": "object"} for c in CLASSES] - old
    categories_m1 = [{"id": c.category_id, "name": c.name, "supercategory": "object"} for c in CLASSES_M1]
    categories_m2 = [{"id": c.category_id, "name": c.name, "supercategory": "object"} for c in CLASSES_M2]

    # counters
    ann_id = 1
    ann_m2_id = 1

    for img_id, (key, img_path, mask_m1_path, mask_m2_path) in enumerate(tqdm(pairs, desc=f"Exporting annotations"), start=1):
        # Link/copy image
        img_name = os.path.basename(img_path)
        out_img_path = os.path.join(out_images_dir, img_name)

        if not os.path.exists(out_img_path):
            if copy_images:
                shutil.copy2(img_path, out_img_path)
            else:
                # symlink if supported; fallback to copy
                try:
                    os.symlink(os.path.abspath(img_path), out_img_path)
                except OSError:
                    shutil.copy2(img_path, out_img_path)

        with Image.open(img_path) as im:
            w, h = im.size

        # COCO Format Image Object
        images.append({"id": img_id, "file_name": img_name, "width": w, "height": h})
        #Image.open(img_path).show()


        # ######################
        #
        # METHOD 1 ====================================================================================
        #
        # ######################

        
        #print("\nMETHOD 1")

        # segmentation image obtained in a previous step
        mask_m1_rgb = np.array(Image.open(mask_m1_path).convert("RGB")) # 3 channels

        # get rid of artefacts
        mask_m1_rgb = clean_rgb_mask(mask_m1_rgb)
 
        # 1) keep *PURE* (mostly) RED 
        facade_mask_m1 = filter_channels(mask_m1_rgb, channels="r")
        
        # 2) binary mask: *ANY* RED pixel = facade (but value of a parameter sometimes can be carried also in GREEN, but WITHIN the RED boundaries)
        facade_mask_m1_filter = facade_mask_m1.copy()
        facade_binary_m1 = (facade_mask_m1_filter[:, :, 0] > 0)

        #Image.fromarray(facade_mask_m1).show()
        #Image.fromarray(facade_binary_m1).show()
 

        # 3) connected components, min_area filters small instances
        facade_instances_m1 = instances_from_binary_mask(facade_binary_m1, min_area=min_facade_area)

        #display_contours(facade_binary_m1, facade_instances_m1)
        
        # 4) write annotations:
        for instance_mask in facade_instances_m1[:]:

            #Image.fromarray(instance_mask).show()
            instance_mask_smooth = smooth_binary_mask(instance_mask)

            rle = coco_rle_from_mask(instance_mask_smooth)
            area = float(mask_utils.area(rle))
            bbox = mask_utils.toBbox(rle).tolist()
            parameter_value = get_parameter_value_from_instance_mask(instance_mask, facade_mask_m1)


            annotations_m1.append({
                "id": ann_id,
                "image_id": img_id,
                "category_id": get_class_id_per_value(parameter_value),     # parameter value
                "segmentation": rle,
                "area": area,
                "bbox": bbox,
                "iscrowd": 0,
                "parameter_value": parameter_value      #just leave it for now, detectron ignores it either way 
            })

            ann_id += 1


        # ######################
        #        
        # METHOD 2 =====================================================================
        #
        # ######################

        #print("\nMETHOD 2")
        
        mask_m2_rgb = np.array(Image.open(mask_m2_path).convert("RGB"))
        mask_m2_rgb = clean_rgb_mask(mask_m2_rgb)
       
       # Find Facades - this time, they are represented in a fixed color value (~255) on a different channel - wheter G or B is arbirtrary, we chose: B
        mask_facades_m2 = filter_channels(mask_m2_rgb, "rb")
        mask_windows_m2 = filter_channels(mask_m2_rgb, "r")

        #Image.fromarray(mask_facades_m2).show()
        #Image.fromarray(mask_windows_m2).show()
        
        windows_binary_m2 = (mask_windows_m2[:, :, 0] > 0)  # RED channel
        facade_binary_m2 = (mask_facades_m2[:,:, 2] > 0)
        #facade_binary_m22 = (mask_facades_m2[:,:, 2] > 0)  + windows_binary_m2  # RED and BLUE channels
        
        #a = np.zeros(shape=facade_binary_m2.shape)
        #a[facade_binary_m2] = 255
        #Image.fromarray(a).show()

        #
        # ######################
        # FACADE ---------------------------------
        # ######################
        #

        # we can use previously found instances while annotating M1
        #facade_instances_m2 = instances_from_binary_mask(facade_binary_m1, min_area=min_facade_area)

        scaling_factor = 0.8
        # 0.8 scaling, because of window openings reduce facade area 
        facade_instances_m2 = instances_from_binary_mask(facade_binary_m2, min_area=scaling_factor*min_facade_area)
        #facade_instances_m22 = instances_from_binary_mask(facade_binary_m22, min_area=min_facade_area)

        facade_instances_ids = []

        #display_contours(facade_binary_m2, facade_instances_m2)

        for idx, facade_instance in enumerate(facade_instances_m2[:]):

            #Image.fromarray(facade_instance).show()
            parameter_value = float(-1)
            facade_instance_smooth = smooth_binary_mask(facade_instance) # binary
            #Image.fromarray(facade_instance_smooth).show()

            rle = coco_rle_from_mask(facade_instance_smooth)
            area = float(mask_utils.area(rle))
            bbox = mask_utils.toBbox(rle).tolist()
 
            #Image.fromarray(facade_instance_smooth).show()
            
            #
            # ######################
            # WINDOWS ---------------------------------
            # ######################
            #

            # outer boundaries
            contour, hierarchy = cv2.findContours(facade_instance_smooth, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
            facade_filter_mask = np.zeros_like(facade_instance_smooth)

            # fill outer contour
            cv2.drawContours(facade_filter_mask, contour, -1, 255, thickness=-1)

            binary_facade_filter_mask = (facade_filter_mask > 0)

            windows_binary_mask_filtered = windows_binary_m2.copy()
            windows_binary_mask_filtered[~binary_facade_filter_mask] = 0

            # mini-optimization - parameter is the same for every window instance (and so is category)
            #Image.fromarray(windows_binary_mask_filtered).show(title="Windows Binary Mask Filtered")
            #Image.fromarray(mask_windows_m2).show(title="Mask Windows M2")

            parameter_value = get_parameter_value_from_instance_mask(windows_binary_mask_filtered, mask_windows_m2)
            if parameter_value == -1:
                continue
            
            # annotations representing a Facade
            annotations_m2.append({
                "id": ann_m2_id,
                "image_id": img_id,
                "category_id": 1000,     # semantics: Facade 
                "segmentation": rle,
                "area": area,
                "bbox": bbox,
                "iscrowd": 0,
                "parameter_value": parameter_value,
                "inside_area": -1
            })
            facade_instances_ids.append(ann_m2_id)
            ann_m2_id += 1

            # window instances in current facade
            windows_instances = instances_from_binary_mask(windows_binary_mask_filtered, min_area=min_window_area)
            windows_category_id = get_class_id_per_value(parameter_value)

            #display_contours(windows_binary_mask_filtered, windows_instances)

            for id_w, windows_instance in enumerate(windows_instances):
                windows_instance_smooth = smooth_binary_mask(windows_instance)
                rle = coco_rle_from_mask(windows_instance_smooth)
                area = float(mask_utils.area(rle))
                bbox = mask_utils.toBbox(rle).tolist()
                #parameter_value = get_parameter_value_from_instance_mask(windows_instance, mask_windows_m2)
  
                # annotations representing a Window
                annotations_m2.append({
                    "id": ann_m2_id,
                    "image_id": img_id,
                    "category_id": windows_category_id,       
                    "segmentation": rle,
                    "area": area,
                    "bbox": bbox,
                    "iscrowd": 0,
                    "parameter_value": parameter_value,
                    "inside_area": facade_instances_ids[-1]     # additional info, but no use currently
                })

                ann_m2_id += 1
        
    #print("ANNOTATIONS M1:")
    #for ann in annotations_m1:
    #    print("A1:", ann)

    #print("ANNOTATIONS M2:")
    #for ann in annotations_m2:
    #    print("A2:", ann)
        
    coco_m1 = {"images": images, "annotations": annotations_m1, "categories": categories_m1}
    coco_m2 = {"images": images, "annotations": annotations_m2, "categories": categories_m2}

    with open(out_json_m1_path, "w", encoding="utf-8") as f:
        json.dump(coco_m1, f, ensure_ascii=False, indent=3)
    with open(out_json_m2_path, "w", encoding="utf-8") as f:
        json.dump(coco_m2, f, ensure_ascii=False, indent=3)

    return


def make_splits(pairs: List[Tuple[str, str, str]], seed: int = 0, train=0.8, val=0.1):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(pairs))
    rng.shuffle(idx)

    n = len(pairs)
    n_train = int(round(train * n))
    n_val = int(round(val * n))
    n_test = n - n_train - n_val

    train_pairs = [pairs[i] for i in idx[:n_train]]
    val_pairs = [pairs[i] for i in idx[n_train:n_train+n_val]]
    test_pairs = [pairs[i] for i in idx[n_train+n_val:]]

    return train_pairs, val_pairs, test_pairs


In [ ]:
print("START", "="*10)

#RAW_DIR = "C:\\Users\\WA\\Desktop\\badanie\\syncity3D\\spheric_photos_and_masks"
#OUT_DIR = "C:\\Users\\WA\\Desktop\\badanie\\syncity3D\\COCO_DATASETS\\D3" 


WORKSPACE_DIR = Path(r"C:\Users\WA\Desktop\badanie") # !! directory with our project and other projects (ESRI.lib)
BASE_DIR = WORKSPACE_DIR / "syncity3D" 
SCRIPTS_DIR = BASE_DIR / "scripts"

# stage 1 - snapshots back, right, front, left, top, bottom 
CE_SNAPSHOTS_DIR = BASE_DIR / "images"
# stage 2 - spheric 'photos' and their segmention/semantic color masks
SPHERIC_PHOTOS_DIR = BASE_DIR / "dataset_stages" / "spheric_photos_and_masks"
# final stage - COCO format ready dataset /images /annotations
COCO_DATASET_DIR = BASE_DIR / "COCO_DATASETS" / "D4" # D4!!!!!

RAW_DIR = SPHERIC_PHOTOS_DIR
OUT_DIR = COCO_DATASET_DIR


# test of 5 images x m-X
#RAW_DIR = "C:\\Users\WA\Desktop\\misc\\sample_dataset_new_dimmed"
#OUT_DIR = "C:\\Users\\WA\\Desktop\\misc\\sample_dataset_out_new_dimmed"

pairs = find_pairs(RAW_DIR)
train_pairs, val_pairs, test_pairs = make_splits(pairs, seed=0, train=0.8, val=0.1)

#print("test_pairs")
#print(test_pairs)

# filter small instances 
min_facade_area = 20000
min_windows_area = 85
copy_images=False


export_split_to_coco(
    train_pairs,
    out_images_dir=os.path.join(OUT_DIR, "images", "train"),
    out_json_m1_path=os.path.join(OUT_DIR, "annotations", "instances_train_m1.json"),
    out_json_m2_path=os.path.join(OUT_DIR, "annotations", "instances_train_m2.json"),
    copy_images=copy_images,
    min_facade_area=min_facade_area,  
    min_window_area=min_windows_area
)

export_split_to_coco(
    val_pairs,
    out_images_dir=os.path.join(OUT_DIR, "images", "val"),
    out_json_m1_path=os.path.join(OUT_DIR, "annotations", "instances_val_m1.json"),
    out_json_m2_path=os.path.join(OUT_DIR, "annotations", "instances_val_m2.json"),
    copy_images=copy_images,
    min_facade_area=min_facade_area, 
    min_window_area=min_windows_area
)

export_split_to_coco(
    test_pairs,
    out_images_dir=os.path.join(OUT_DIR, "images", "test"),
    out_json_m1_path=os.path.join(OUT_DIR, "annotations", "instances_test_m1.json"),
    out_json_m2_path=os.path.join(OUT_DIR, "annotations", "instances_test_m2.json"),
    copy_images=copy_images,
    min_facade_area=min_facade_area,
    min_window_area=min_windows_area
)

# Zapis splitu dla reprodukowalności
split_path = os.path.join(OUT_DIR, "split_keys.json")
with open(split_path, "w", encoding="utf-8") as f:
    json.dump({
        "train": [k for (k, _, _, _) in train_pairs],
        "val": [k for (k, _, _, _) in val_pairs],
        "test": [k for (k, _, _, _) in test_pairs],
    }, f, ensure_ascii=False, indent=2)


print("DONE", "="*10)

START ==========


Exporting annotations:   0%|          | 0/1 [00:00<?, ?it/s]


METHOD 2


Exporting annotations: 100%|██████████| 1/1 [04:31<00:00, 271.08s/it]

DONE ==========
